# 15 — Submission de la configuración final candidata

Mismo esquema que `14_submission_base.ipynb`, con la **configuración final** que salió de la última ronda de features (`laboratorio.ipynb`, prueba P6b): CatBoost con las categóricas crudas + modelo de texto + numéricas. CV: **0,9365 ± 0,0017** (semilla 42) y 0,9359 ± 0,0019 (semilla 7), contra 0,9339 / 0,9332 de la Base.

Todo sale de `features.py` (`armar_pipeline_final`, `INDICES_CAT` y las funciones de features), así que esta notebook no define nada propio.

No toca `train.ipynb` ni `predict.ipynb`. **No sube nada**: el CSV lo subo a mano.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

from features import (
    aplanar_anidadas, agregar_features_contenido, agregar_categoricas_crudas,
    armar_pipeline_final, INDICES_CAT,
)

## 1. Datos

A train y test se les aplican **las mismas tres funciones determinísticas**: `aplanar_anidadas`, `agregar_features_contenido` y `agregar_categoricas_crudas`.

In [2]:
train = pd.read_json("../data/train_data.jsonlines", lines=True)
test = pd.read_json("../data/test_data.jsonlines", lines=True)
y = (train["condition"] == "new").astype(int)

train = agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(train)))
test = agregar_categoricas_crudas(agregar_features_contenido(aplanar_anidadas(test)))

print("train:", train.shape, "| test:", test.shape)

train: (70000, 78) | test: (30000, 77)


## 2. Ajuste con todo el train

`armar_pipeline_final()` arma el pipeline y `INDICES_CAT` le dice a CatBoost qué columnas son categóricas. Va en el `fit`, no en el constructor: es una incompatibilidad de CatBoost con `clone` de sklearn.

In [3]:
pipeline_final = armar_pipeline_final()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pipeline_final.fit(train, y, modelo__cat_features=INDICES_CAT)

print("Categóricas que usa CatBoost (índices):", pipeline_final.named_steps["modelo"].get_cat_feature_indices())

Categóricas que usa CatBoost (índices): [525, 526, 527, 528, 529, 530]


## 3. Predicción y CSV

Formato confirmado por Kaggle: `ID` = número de fila de 1 a 30.000 (orden de `test_data.jsonlines`), `condition` = `"new"`/`"used"`.

In [4]:
prediccion = pipeline_final.predict(test)

submission = pd.DataFrame({
    "ID": np.arange(1, len(test) + 1),
    "condition": pd.Series(prediccion).map({1: "new", 0: "used"}),
})

Path("../submission").mkdir(exist_ok=True)
archivo = "../submission/submission_catboost.csv"
submission.to_csv(archivo, index=False)
print("Guardado:", archivo)
submission.head()

Guardado: ../submission/submission_catboost.csv


,ID,condition
0,1,used
1,2,new
2,3,used
3,4,used
4,5,new


## 4. Chequeos

Asserts de formato sobre el CSV leído desde el disco, y dos chequeos de sentido común: % de `new` predicho y cuántas predicciones cambian contra la submission de la Base (`submission_base_0934.csv`, público 0,93476). Si la CV mejoró ~0,003, deberían cambiar algunos cientos de filas, no miles.

In [5]:
leida = pd.read_csv(archivo)

assert leida.shape == (30000, 2), leida.shape
assert list(leida.columns) == ["ID", "condition"], list(leida.columns)
assert (leida["ID"].to_numpy() == np.arange(1, 30001)).all()
assert leida["condition"].isin(["new", "used"]).all()
assert leida["condition"].notna().all()
print("Formato OK")

print("% new predicho en test:", round((leida["condition"] == "new").mean(), 4), "| % new en train:", round(y.mean(), 4))

base = pd.read_csv("../submission/submission_base_0934.csv")
cambian = (base["condition"] != leida["condition"]).sum()
print("Predicciones distintas vs submission_base_0934.csv:", cambian, f"({cambian / len(leida):.1%})")

Formato OK
% new predicho en test: 0.5286 | % new en train: 0.537
Predicciones distintas vs submission_base_0934.csv: 588 (2.0%)
